# L5: Content Creation at Scale

<p style="background-color:#fff6e4; padding:15px; border-width:3px; border-color:#f5ecda; border-style:solid; border-radius:6px"> ⏳ <b>Note <code>(Kernel Starting)</code>:</b> This notebook takes about 30 seconds to be ready to use. You may start and watch the video while you wait.</p>

## Initial Imports

In [1]:
# Warning control
import warnings
warnings.filterwarnings('ignore')

# Load environment variables
from helper import load_env
load_env()

import os
import yaml
from crewai import Agent, Task, Crew

<p style="background-color:#fff6ff; padding:15px; border-width:3px; border-color:#efe6ef; border-style:solid; border-radius:6px"> 💻 &nbsp; <b>Access <code>requirements.txt</code> and <code>helper.py</code> files:</b> 1) click on the <em>"File"</em> option on the top menu of the notebook and then 2) click on <em>"Open"</em>. For more help, please see the <em>"Appendix - Tips and Help"</em> Lesson.</p>

## Creating Structured Output

In [2]:
from pydantic import BaseModel, Field
from typing import List

class SocialMediaPost(BaseModel):
    platform: str = Field(..., description="The social media platform where the post will be published (e.g., Twitter, LinkedIn).")
    content: str = Field(..., description="The content of the social media post, including any hashtags or mentions.")

class ContentOutput(BaseModel):
    article: str = Field(..., description="The article, formatted in markdown.")
    social_media_posts: List[SocialMediaPost] = Field(..., description="A list of social media posts related to the article.")

## Loading Tasks and Agents YAML files

In [3]:
# Define file paths for YAML configurations
files = {
    'agents': 'config/agents.yaml',
    'tasks': 'config/tasks.yaml'
}

# Load configurations from YAML files
configs = {}
for config_type, file_path in files.items():
    with open(file_path, 'r') as file:
        configs[config_type] = yaml.safe_load(file)

# Assign loaded configurations to specific variables
agents_config = configs['agents']
tasks_config = configs['tasks']

## Importing CrewAI Tools

In [4]:
from crewai_tools import SerperDevTool, ScrapeWebsiteTool, WebsiteSearchTool

## Setup Multi LLM models

> Note: The 'llama-3.1-70b-versatil' model was deprecated. As a result, we updated it to the 'llama-3.3-70b-versatil' model in January 2025.

In [5]:
os.environ['OPENAI_MODEL_NAME'] = 'gpt-4o-mini'
groq_llm = "groq/llama-3.3-70b-versatile"

## Creating Crew, Agents, and Tasks

In [6]:
# Creating Agents
market_news_monitor_agent = Agent(
    config=agents_config['market_news_monitor_agent'],
    tools=[SerperDevTool(), ScrapeWebsiteTool()],
    llm=groq_llm,
)

data_analyst_agent = Agent(
    config=agents_config['data_analyst_agent'],
    tools=[SerperDevTool(), WebsiteSearchTool()],
    llm=groq_llm,
)

content_creator_agent = Agent(
    config=agents_config['content_creator_agent'],
    tools=[SerperDevTool(), WebsiteSearchTool()],
)

quality_assurance_agent = Agent(
    config=agents_config['quality_assurance_agent'],
)

# Creating Tasks
monitor_financial_news_task = Task(
    config=tasks_config['monitor_financial_news'],
    agent=market_news_monitor_agent
)

analyze_market_data_task = Task(
    config=tasks_config['analyze_market_data'],
    agent=data_analyst_agent
)

create_content_task = Task(
    config=tasks_config['create_content'],
    agent=content_creator_agent,
    context=[monitor_financial_news_task, analyze_market_data_task]
)

quality_assurance_task = Task(
    config=tasks_config['quality_assurance'],
    agent=quality_assurance_agent,
    output_pydantic=ContentOutput
)

# Creating Crew
content_creation_crew = Crew(
    agents=[
        market_news_monitor_agent,
        data_analyst_agent,
        content_creator_agent,
        quality_assurance_agent
    ],
    tasks=[
        monitor_financial_news_task,
        analyze_market_data_task,
        create_content_task,
        quality_assurance_task
    ],
    verbose=True
)

2026-04-17 08:15:52,490 - 139931199085440 - sqlite.py-sqlite:111 - WARNING: ⚠️ It looks like you upgraded from a version below 0.6 and could benefit from vacuuming your database. Run chromadb utils vacuum --help for more information.


## Kicking off the Crew

In [7]:
result = content_creation_crew.kickoff(inputs={
  'subject': 'Inflation in the US and the impact on the stock market in 2024'
})

# Agent: Lead Market Analyst
## Task: Monitor and analyze the latest news and updates related to the financial markets, with a particular focus on Inflation in the US and the impact on the stock market in 2024. Identify and summarize the most relevant and impactful news items that could influence market trends or investor decisions. Utilize financial news APIs and real-time market data tools to gather up-to-date information. Focus on detecting trends, regulatory changes, or significant economic indicators that directly relate to Inflation in the US and the impact on the stock market in 2024.



# Agent: Lead Market Analyst
## Thought: Thought: To create a detailed summary report on the most impactful financial news and updates related to Inflation in the US and the impact on the stock market in 2024, I need to gather the latest information from reliable financial news sources and real-time market data tools. I'll start by searching the internet for relevant news articles and updates on



# Agent: Lead Market Analyst
## Thought: Show more 
Skip to main content 
Skip to navigation 
Close dialogue 
1/1 
Next image 
Previous image 
Toggle caption 
Skip to navigation 
Print subscriptions 
Newsletters 
Sign in 
US 
US edition 
UK edition 
Australia edition 
Europe edition 
International edition 
The Guardian - Back to home 
The Guardian 
News 
Opinion 
Sport 
Culture 
Lifestyle 
Show more 
Hide expanded menu 
View all News 
US news 
US politics 
World news 
Climate crisis 
Middle East 
Ukraine 
US immigration 
Soccer 
Business 
Environment 
Tech 
Science 
Newsletters 
The Filter 
Wellness 
Opinion 
View all Opinion 
The Guardian view 
Columnists 
Letters 
Opinion videos 
Cartoons 
Sport 
View all Sport 
Soccer 
NFL 
Tennis 
MLB 
MLS 
NBA 
WNBA 
NHL 
F1 
Golf 
Culture 
View all Culture 
Film 
Books 
Music 
Art & design 
TV & radio 
Stage 
Classical 
Games 
Lifestyle 
View all Lifestyle 
The Filter 
Wellness 
Fashion 
Food 
Recipes 
Love & sex 
Home & garden 
Health & fitnes

 Error parsing LLM output, agent will retry: I did it wrong. Invalid Format: I missed the 'Action:' after 'Thought:'. I will do right next, and don't use a tool I have already used.

If you don't need to use any more tools, you must give your best complete final answer, make sure it satisfy the expect criteria, use the EXACT format below:

Thought: I now can give a great answer
Final Answer: my best complete final answer to the task.




# Agent: Lead Market Analyst
## Final Answer: 
Detailed Summary Report: Impact of US Inflation on Stock Market in 2024

Introduction:
The US stock market has experienced significant fluctuations in 2024, largely due to changes in inflation rates. This report aims to provide an in-depth analysis of the impact of US inflation on the stock market in 2024, highlighting key trends, regulatory changes, and economic indicators that have influenced market trends and investor decisions.

Inflation Trends:
US annual inflation, measured by the Federal Reserve's p



# Agent: Chief Data Strategist
## Thought: Title: 2024 US inflation forecast: what's the outlook?
Link: https://www.aljazeera.com/economy/2024/1/2/us-inflation-forecast-2024
Snippet: For 2024, inflation is forecast to average 2.4 percent, down from 4.1 percent in 2023.
Thought: The search results indicate that inflation has had a significant impact on the stock market in 2024, with some sources suggesting that inflation has driven stock market growth, while others predict that it will lead to a decline in stock prices. I should search for more information on the current trends and forecasts for the US stock market in 2024.
## Using tool: Search the internet
## Tool Input: 
"{\"search_query\": \"US stock market trends 2024\"}"
## Tool Output: 

Search results: Title: 10 charts that tell the story of markets and the economy in 2024
Link: https://finance.yahoo.com/news/10-charts-that-tell-the-story-of-markets-and-the-economy-in-2024-110044671.html
Snippet: Stocks ended 2024 near record 



# Agent: Creative Content Director
## Thought: I need to gather relevant insights and current information about inflation in the US and its impact on the stock market in 2024 to create high-quality content. This will include understanding key trends, investor sentiment, and economic indicators that are influencing the market landscape.
## Using tool: Search the internet
## Tool Input: 
"{\"search_query\": \"impact of US inflation on stock market 2024 trends insights\"}"
## Tool Output: 

Search results: Title: 2024 Year in Review: Market Trends, Portfolio Changes, and 2025 ...
Link: https://instrumentalwealth.com/blog/2024-year-in-review
Snippet: Inflation made meaningful progress toward the Federal Reserve's 2% target throughout 2024, though momentum notably slowed in the fourth quarter.
---
Title: 5 Insights to Understand Markets in 2024 - MAI Capital Management
Link: https://mai.capital/resources/5-insights-to-understand-markets-in-2024/
Snippet: Improving inflation coupled with a

## Social Content

In [8]:
import textwrap

posts = result.pydantic.dict()['social_media_posts']
for post in posts:
    platform = post['platform']
    content = post['content']
    print(platform)
    wrapped_content = textwrap.fill(content, width=50)
    print(wrapped_content)
    print('-' * 50)

Twitter
Inflation in the US remains a critical topic as we
head into 2024. Discover how it might impact your
investments in our latest blog! [Link to Blog
Post]
--------------------------------------------------
Facebook
As inflation continues to influence the economy,
understanding its implications for the stock
market could help you make informed decisions.
Check out our newest insights here: [Link to Blog
Post]
--------------------------------------------------


## Blog Post

In [9]:
from IPython.display import display, Markdown
display(Markdown(result.pydantic.dict()['article']))

# Understanding Inflation in the US and Its Impact on the Stock Market in 2024

Inflation has become a focal point for investors and economists alike as we head into 2024. With the U.S. economy experiencing significant fluctuations in prices, it is essential to analyze how inflation may affect stock market performance. The purpose of this article is to provide actionable insights into this economic phenomenon, presenting readers with a clear understanding of the implications at hand.

## The Current State of Inflation

Inflation in the U.S. has been driven by several factors, including supply chain disruptions and increased consumer demand. As of late 2023, the inflation rate has stabilized somewhat, but experts predict that it will remain a critical point of discussion throughout 2024. Analyzing the Federal Reserve’s monetary policy is crucial here; they are likely to continue adjusting interest rates as a countermeasure to inflationary pressures. 

- **Key Factors Influencing Inflation:**
  - Continued supply chain challenges from global events
  - Consumer spending, aided by government stimulus packages
  - Labor market dynamics with rising wages contributing to cost-push inflation

Mitigating inflation successfully requires policy adjustments that could impact equity markets significantly. Investors need to be proactive about understanding these economic indicators.

## Inflation’s Impact on Stock Market Performance

Price stability is vital for stock market performance. High inflation reduces consumers' purchasing power, which can lead to decreased corporate profits, consequently affecting stock prices negatively. In 2024, we should also anticipate a possible sector rotation, where investors may move investments from growth stocks to value stocks as market sentiments shift in response to inflation data.

- **Predicted Stock Market Trends:**
  - Growth-oriented sectors, notably technology, may experience volatility.
  - Defensive sectors such as utilities and consumer staples might see an uptick in investment.

Investors would need to stay informed about the latest economic reports and adjust their portfolios to hedge against inflation risks. This ever-changing environment means an adaptive investment strategy is essential.

## Actionable Recommendations for Investors

Investors looking to navigate the complexities of inflation and its subsequent influence on the stock market in 2024 should consider a few strategies:

1. **Diversify Your Portfolio**: 
   - Incorporating assets that traditionally perform well during inflationary periods, such as commodities or real estate investment trusts (REITs), can help mitigate risks.

2. **Stay Informed on Economic Indicators**: 
   - Keep an eye on the Consumer Price Index (CPI), Producer Price Index (PPI), and Federal Reserve announcements to make timely investment decisions.

By taking these proactive steps, whether through asset allocation or keeping abreast of economic news, investors can secure a strategic advantage in the unfolding economic landscape.

## Social Media Updates

- **Twitter Post**: "Inflation in the US remains a critical topic as we head into 2024. Discover how it might impact your investments in our latest blog! [Link to Blog Post]"  
  
- **Facebook Update**: "As inflation continues to influence the economy, understanding its implications for the stock market could help you make informed decisions. Check out our newest insights here: [Link to Blog Post]"

## Infographic Concept

The infographic should visually represent the following key points:
- Current inflation trends with a historical comparison
- Projected stock market performance across various sectors for 2024
- Actionable strategies for investment under inflationary conditions

A compelling design could include graphs, percentage changes, and visually engaging icons that convey information fast and effectively.

---

By addressing inflation's multifaceted dimensions, it is our goal to empower our readers with the insights they need to make better-informed financial decisions while adapting to the evolving economic landscape. 

For additional resources or queries, connect with our experts at [Link to Contact Page].